## Part e) visualisation

### Outline for part e (TEMP)

**Checklist:**
*Written in project description:*
- Test train split
    - use cross-validation for training, test-split for validation
- tuning of hyperparams
- Cite dataset
- Accuracy score
- Discuss:
    - varius params
    - hyperparams
        - learning rates
        - regularization param $\lambda$
        - activation functs
        - nr of hidden layers
        - nodes
- Compare with Scikit-learn, TensorFlow/Keras or PyTorch
- Compare nn with logistic regression(no hidden layer). LG is *baseline* comparison
- Confusion matrix
- Report precision, recall and $F_1$ score. CF makes these interpretable. Use sklearn for this

*My own points:*
- Make at least three nn's with various activation fucntions and nodes per hidden layer



### LLM usage notes

neural_network.py 
    - lvl 1 GPT-6 Luna impelenting softmax multiclass cross entropy derivative

### Notes from 24.oct 

#### Installing tensorflow 
To access the mnist dataset, use tensorflow. Be sure to check which installation to use as it depends on operative system and if you have a dedicated GPU. I (Hannah) do not have a dedicated GPU, so i am using the CPU install as detailed here: https://www.tensorflow.org/install/pip#cpu
The same wepbage shows installation for MacOS and such 

In [5]:
from general_functions import MnistImportAndScale
import numpy as np 
import matplotlib.pyplot as plt

from _Part_a import ReLU, LeakyReLU, Sigmoid, Identity, Softmax, MulticlassCrossEntropy
from optimizer import Adam, train
from neural_network import NeuralNetwork
from general_functions import RungeData, Scaler, FIG_DIR, save_best_lr, load_best_lr, ArgsHelper


mnist_data = MnistImportAndScale()
x_train, x_test, y_train, y_test = mnist_data.return_data()
cv_splits = mnist_data.kfold_splits(n_splits=2)

##### Outline for tomorrow
- add kfold cv in tuning of hyperparams loop
- decide layers and layer sizes
- add function for saving fully trained models

---
#### Notes taken during tuesday lecture week 44
**Tuesday lecutre contains a bunch of good info on what to report in the report**

1. For tuning loop - only cost neccecary to be logged. Save best hyperparams
     - sklearn GridSearchCV 
        - what scoring? multiclass cross entropy
        - replace optimizer (like ridge linear regression ) with nn
            - challenging part will be figuring out how to define the pipeline prob

        - could prob reuse most of cv code from proj1

    - score cost, mean and std
        - perhaps present in a table if relevant
            - may be difficult due to several hyperparams being trained. 
        - lots of good examples on how to present in lecture 

2. use best hyperparams to train fresh model on all train data, test against test set with cost. 
    - Present best model with cv cost(from 1. ?? a bit unclear. page 12. ask), std and test cost **in a table**
    - no need to do cv here

3. With model trained only on training data, report ROC, CM and so on

---
ask about cv decomposition  -> proj1 final figs

In [ ]:
# Implement kfold cv during training

gammas = np.logspace(-4, 0, 4)
batches = 16

activations = {'ReLU': ReLU(), 'LeakyReLU': LeakyReLU()}
results = {}

y_train_1hot = np.eye(10)[np.asarray(y_train).reshape(-1).astype(int)] #converting to one-hot target vector

for i, (activation_name, activation) in enumerate(activations.items()):
    for j, gamma in enumerate(gammas):
        fold_train_costs = []
        fold_val_costs = []

        for fold, (train_idx, val_idx) in enumerate(cv_splits):
            X_fold_train, y_fold_train = x_train[train_idx], y_train_1hot[train_idx]
            X_fold_val, y_fold_val = x_train[val_idx], y_train_1hot[val_idx]

            # A new model and optimizer for every fold.
            nn = NeuralNetwork(
                784, [50, 10], [activation, Softmax()],
                cost=MulticlassCrossEntropy(), #Multiclass entropy as we have four classes
                seed=2026 + fold,
            )
            history = train(
                nn, X_fold_train, y_fold_train,
                Adam(gamma), epochs=150, batch_size=16,
                seed=2026 + fold,
                X_val=X_fold_val, y_val=y_fold_val,
            )

            fold_train_costs.append(history["train"][-1])
            fold_val_costs.append(history["val"][-1])


        results[(activation_name, gamma)] = {
            "mean_train_cost": np.mean(fold_train_costs),
            "mean_val_cost": np.mean(fold_val_costs),
        }

        print(
            f"{activation_name}, gamma={gamma:.1e}: "
            f"mean train cost={np.mean(fold_train_costs):.4f}, "
            f"mean validation cost={np.mean(fold_val_costs):.4f}"
        )


ReLU, gamma=1.0e-04: mean train cost=62.3113, mean validation cost=5370.8544
ReLU, gamma=2.2e-03: mean train cost=170.2761, mean validation cost=16536.2129
ReLU, gamma=4.6e-02: mean train cost=20788.5174, mean validation cost=30802.9964
ReLU, gamma=1.0e+00: mean train cost=71888.2051, mean validation cost=71735.9500
LeakyReLU, gamma=1.0e-04: mean train cost=68.5064, mean validation cost=5285.6501


In [ ]:
import json
from general_functions import RESULTS_DIR
with open(RESULTS_DIR / "MNIST_lrs", "w") as f:
    json.dump(results, f, indent=2)